In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
attention_size = 32
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

# train stays on cpu (unfold is a view, moving it would copy every window), batches move to device
x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)

In [4]:
# module declaration
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len):
        super().__init__()

        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.position_embedding = nn.Embedding(num_embeddings=sequence_len, embedding_dim=embedding_size)

    def forward(self, input):
        return self.token_embedding(input=input) + self.position_embedding(torch.arange(input.size(dim=1), device=input.device))

class SelfAttention(nn.Module):
    def __init__(self, embedding_size, attention_size, sequence_len):
        super().__init__()

        self.attention_size = attention_size
        self.query = nn.Linear(in_features=embedding_size, out_features=attention_size, bias=False)
        self.key = nn.Linear(in_features=embedding_size, out_features=attention_size, bias=False)
        self.value = nn.Linear(in_features=embedding_size, out_features=attention_size, bias=False)

        tril = torch.tril(input=torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, input):
        query = self.query(input=input)
        key = self.key(input=input)
        value = self.value(input=input)

        T = input.size(dim=1)

        scores = query @ key.transpose(dim0=-2, dim1=-1) * self.attention_size ** -0.5
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)

        return weights @ value

In [5]:
# model declaration
class Model(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len, attention_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            TokenAndPositionEmbedding(vocab_size=vocab_size, embedding_size=embedding_size, sequence_len=sequence_len),
            SelfAttention(embedding_size=embedding_size, attention_size=attention_size, sequence_len=sequence_len),
            nn.Linear(in_features=attention_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=vocab_size),
        )

    def forward(self, input):
        return self.layers(input=input)

In [6]:
# model initialization
model = Model(vocab_size=vocab_size, embedding_size=embedding_size, sequence_len=sequence_len, attention_size=attention_size, hidden_size=hidden_size).to(device=device)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 32353


In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch].to(device=device)
    y_batch = y_train[batch].to(device=device)
    train_logits = model(input=x_batch)
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.155622959136963, val loss = 4.157437801361084
Step 1000: train loss = 2.3385987281799316, val loss = 2.3491852283477783
Step 2000: train loss = 1.9680943489074707, val loss = 2.130216598510742
Step 3000: train loss = 1.9539287090301514, val loss = 2.061728000640869
Step 4000: train loss = 1.8780485391616821, val loss = 2.0192224979400635
Step 4999: train loss = 1.8144619464874268, val loss = 2.013174533843994


In [8]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long, device=device)

    while len(output) < max_output_length:
        logits = model(input=i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long, device=device)

In [9]:
print(''.join(output))

ROMEO:
If is,
And sither sairstyful my man! that
Bluck tre lookp! Some, and,
Fooman:
O like acome to me gor lady, the be dispent:
To befort
Cone nhou slwor:
Dour we busand than fice

Agault thoundsion the behoose a 'gaing
Han this by and ude,
Sall me answe nor to me guent in all'd tak amby lothosse here whely a gue popley.

GLOUCESTER:
Is Macighod daign h, whe ley th, Ath thip easgue trude bone From you, nowine diviltest lieven,
Sife,
Will come, of lirsy, not;
Wh spected ut hums be falf, be Yorn't! augan the but constime?

BOLINGBROKE:
I
LEONTES:
Uprith hon therm:
Wee a morrown!
Nupt.

SeOr to Yof nige,
Hade hist the perincell ther awat from abe,
Not,
Fir Staugh;
An my diend maning shy, hort
Warwice,
And the doty th eye but boyur rcongs.

KING RICHARINCENTIO:
Ist
Shent;
Butf the us, anshed for iffight,' have to dowk,
Whown:
A muper.

Withe wather on that,
Emide tithe?
-
NTIO:
Well he'embunes slep and mill.

MENES:
Hersond seepist?

Speaking, Lor awose quithen longly
To loven't the stre